In [ ]:
!pip install pandas tabulate pymongo pandas tqdm matplotlib nltk seaborn --quiet

In [5]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'D:\\KULIAH\\SEMESTER 6\\Big Data\\UAS\\.venv\\Lib\\site-packages\\~orch\\lib\\asmjit.dll'
Check the permissions.


[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cu118
  Obtaining dependency information for torchvision from https://download.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for torchaudio from https://download.pytorch.org/whl/cu118/torchaudio-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for torch from https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata
   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   - -------------------------------------- 0.1/5.5 MB 2.8 MB/s eta 0:00:02
   - -------------------------------------- 0.3/5.5 MB 2.7 MB/s eta 0:00:02
   --- ------------------------------------ 0.4/5.5 MB 2.9 MB/s eta 0:00:02
   --- ------------------------------------ 0.4/5.5 MB 2.9 MB/s eta 0:00:02
   --- ------------------------------------ 0.5/5.5 MB 2.3 MB/s eta 0:00:03
   --- ------------------------------

In [ ]:
import pymongo
from pymongo import MongoClient
import re
from tqdm import tqdm
import tqdm
from datetime import datetime
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
import gradio as gr


In [9]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
# Connect to MongoDB
try:
    client = MongoClient('mongodb://localhost:27017')
    client.admin.command('ping')
    print("Connected to MongoDB successfully.")
    db = client['arxiv_db']
    collection1 = db['170k_papers']
    collection2 = db['170k_papers_processed']
except pymongo.errors.ConnectionFailure as e:
    print(f"Could not connect to MongoDB: {e}")
    exit(1)

Connected to MongoDB successfully.


In [3]:
# Count total documents in the collection
total_docs = collection1.count_documents({})
print(f"Total dokumen dalam MongoDB 170k_papers collection: {total_docs}")

Total dokumen dalam MongoDB 170k_papers collection: 179222


In [5]:
# Show all keys in collection
def get_all_keys(collection, sample_size=1000):
    keys = set()
    cursor = collection1.find({}, limit=sample_size)
    for doc in cursor:
        keys.update(doc.keys())
    return keys

all_keys = get_all_keys(collection1)
print("Fields dalam collection:")
print(all_keys)


Fields dalam collection:
{'_id', 'categories', 'title', 'updated', 'primary_category', 'authors', 'pdf_url', 'is_english', 'id', 'published', 'summary'}


In [6]:
# Show one sample data from collection
print("Sample Data from Collection")
sample_doc = collection1.find_one({})
if sample_doc:
    print("Struktur Dokumen:")
    for key, value, in sample_doc.items():
        if isinstance(value, str) and len(value) > 100:
            print(f"{key}: {value[:100]}...")
        else:
            print(f"{key}: {value}")
else:
    print("Tidak ada data dalam collection")

Sample Data from Collection
Struktur Dokumen:
_id: 6853f700fcdfd17d7f675d55
id: http://arxiv.org/abs/2001.12004v2
title: Neural MMO v1.3: A Massively Multiagent Game Environment for Training and Evaluating Neural Networks
authors: Joseph Suarez, Yilun Du, Igor Mordatch, Phillip Isola
summary: Progress in multiagent intelligence research is fundamentally limited by the
number and quality of e...
published: 2020-01-31
updated: 2020-04-17
primary_category: cs.LG
categories: cs.LG, cs.AI, cs.MA, stat.ML
pdf_url: http://arxiv.org/pdf/2001.12004v2
is_english: True


In [ ]:
# Pre-processing text data
class MongoDBPreprocessor:
    def __init__(self, collection):
        self.collection = collection

    def preprocess_text(self, text):
        if not text or pd.isna(text):
            return ""

        text = str(text).lower()  

        # Hapus URL dan email
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = re.sub(r'\S+@\S+', '', text)

        # Hapus karakter aneh, tapi pertahankan tanda baca penting
        text = re.sub(r'[^a-z0-9\s.,!?]', ' ', text)

        # Hapus whitespace berlebih
        text = ' '.join(text.split())

        return text.strip()
    
    def process_sample(self, limit=1):
        """Process sample data for testing"""
        print(f" Processing {limit} sample documents...")
        
        cursor = self.collection.find({}).limit(limit)
        results = []
        
        for doc in cursor:
            original_title = doc.get('title', '')
            original_summary = doc.get('summary', '')
            
            processed_title = self.preprocess_text(original_title)
            processed_summary = self.preprocess_text(original_summary)
            combined_text = f"{processed_title} {processed_summary}".strip()
            
            result = {
                '_id': doc['_id'],
                'original_title': original_title,
                'processed_title': processed_title,
                'original_summary': original_summary[:200] + "..." if len(original_summary) > 200 else original_summary,
                'processed_summary': processed_summary[:200] + "..." if len(processed_summary) > 200 else processed_summary,
                'combined_text': combined_text[:300] + "..." if len(combined_text) > 300 else combined_text,
                'text_length': len(combined_text.split())
            }
            results.append(result)
        
        return results

# Jalankan preprocessing sample
preprocessor = MongoDBPreprocessor(collection1)
sample_results = preprocessor.process_sample(limit=1)

df_sample = pd.DataFrame(sample_results)

pd.set_option("display.max_colwidth", None)

df_sample


 Processing 1 sample documents...


,_id,original_title,processed_title,original_summary,processed_summary,combined_text,text_length
0,6853f700fcdfd17d7f675d55,Neural MMO v1.3: A Massively Multiagent Game Environment for Training and Evaluating Neural Networks,neural mmo v1.3 a massively multiagent game environment for training and evaluating neural networks,"Progress in multiagent intelligence research is fundamentally limited by the\nnumber and quality of environments available for study. In recent years,\nsimulated games have become a dominant research pl...","progress in multiagent intelligence research is fundamentally limited by the number and quality of environments available for study. in recent years, simulated games have become a dominant research pl...","neural mmo v1.3 a massively multiagent game environment for training and evaluating neural networks progress in multiagent intelligence research is fundamentally limited by the number and quality of environments available for study. in recent years, simulated games have become a dominant research pl...",167


In [ ]:
class MongoDBPreprocessor:
    def __init__(self, source_collection, target_collection):
        self.source = source_collection
        self.target = target_collection

    def preprocess_text(self, text):
        if not text or pd.isna(text):
            return ""

        text = str(text).lower()
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = re.sub(r'\S+@\S+', '', text)
        text = re.sub(r'[^a-z0-9\s.,!?]', ' ', text)
        text = ' '.join(text.split())
        return text.strip()

    def process_all_documents(self, batch_size=500):
        total_docs = self.source.count_documents({})
        print(f"Memulai preprocessing untuk {total_docs} dokumen...")
        pbar = tqdm(total=total_docs)

        for i in range(0, total_docs, batch_size):
            batch = list(self.source.find({}).skip(i).limit(batch_size))
            processed_batch = []

            for doc in batch:
                processed_title = self.preprocess_text(doc.get('title', ''))
                processed_summary = self.preprocess_text(doc.get('summary', ''))
                combined_text = f"{processed_title}. {processed_summary}".strip()

                processed_doc = {
                    '_id': doc['_id'],  
                    'id': doc.get('id'),
                    'title': doc.get('title'),
                    'summary': doc.get('summary'),
                    'processed_title': processed_title,
                    'processed_summary': processed_summary,
                    'combined_text': combined_text,
                    'text_length': len(combined_text.split()),
                    'primary_category': doc.get('primary_category'),
                    'categories': doc.get('categories'),
                    'published': doc.get('published'),
                    'updated': doc.get('updated'),
                    'authors': doc.get('authors'),
                    'pdf_url': doc.get('pdf_url'),
                    'is_english': doc.get('is_english'),
                    'is_processed': True
                }

                processed_batch.append(processed_doc)

            if processed_batch:
                try:
                    self.target.insert_many(processed_batch, ordered=False)
                except pymongo.errors.BulkWriteError as bwe:
                    print("Beberapa dokumen mungkin sudah ada di target dan dilewati.")
            
            pbar.update(len(batch))

        pbar.close()
        print(f"Preprocessing selesai untuk {total_docs} dokumen dan disimpan ke koleksi baru.")


def connect_to_mongodb(uri="mongodb://localhost:27017", db_name="arxiv_db"):
    client = MongoClient(uri)
    db = client[db_name]
    return db

db = connect_to_mongodb()
source_collection = db["170k_papers"]
target_collection = db["170k_papers_processed"]

# Jrun preprocessing on the source collection and save to target collection
preprocessor = MongoDBPreprocessor(source_collection, target_collection)
preprocessor.process_all_documents(batch_size=500)


Memulai preprocessing untuk 179222 dokumen...


100%|██████████| 179222/179222 [01:01<00:00, 2934.85it/s]

Preprocessing selesai untuk 179222 dokumen dan disimpan ke koleksi baru.


In [36]:
# Show 3 samples from the processed collection
samples = list(collection2.find({"is_processed": True}).limit(3))

results = []
for doc in samples:
    result = {
        "_id": doc["_id"],
        "original_title": doc.get("title", ""),
        "processed_title": doc.get("processed_title", ""),
        "original_summary": doc.get("summary", "")[:200] + "..." if len(doc.get("summary", "")) > 200 else doc.get("summary", ""),
        "processed_summary": doc.get("processed_summary", "")[:200] + "..." if len(doc.get("processed_summary", "")) > 200 else doc.get("processed_summary", ""),
        "combined_text": doc.get("combined_text", "")[:300] + "..." if len(doc.get("combined_text", "")) > 300 else doc.get("combined_text", ""),
        "text_length": doc.get("text_length", 0)
    }
    results.append(result)

df_samples = pd.DataFrame(results)


pd.set_option("display.max_colwidth", None)

df_samples

,_id,original_title,processed_title,original_summary,processed_summary,combined_text,text_length
0,6853f700fcdfd17d7f675d55,Neural MMO v1.3: A Massively Multiagent Game Environment for Training and Evaluating Neural Networks,neural mmo v1.3 a massively multiagent game environment for training and evaluating neural networks,"Progress in multiagent intelligence research is fundamentally limited by the\nnumber and quality of environments available for study. In recent years,\nsimulated games have become a dominant research pl...","progress in multiagent intelligence research is fundamentally limited by the number and quality of environments available for study. in recent years, simulated games have become a dominant research pl...","neural mmo v1.3 a massively multiagent game environment for training and evaluating neural networks. progress in multiagent intelligence research is fundamentally limited by the number and quality of environments available for study. in recent years, simulated games have become a dominant research p...",167
1,6853f700fcdfd17d7f675d56,Deontological Ethics By Monotonicity Shape Constraints,deontological ethics by monotonicity shape constraints,"We demonstrate how easy it is for modern machine-learned systems to violate\ncommon deontological ethical principles and social norms such as ""favor the\nless fortunate,"" and ""do not penalize good attri...","we demonstrate how easy it is for modern machine learned systems to violate common deontological ethical principles and social norms such as favor the less fortunate, and do not penalize good attribut...","deontological ethics by monotonicity shape constraints. we demonstrate how easy it is for modern machine learned systems to violate common deontological ethical principles and social norms such as favor the less fortunate, and do not penalize good attributes. we propose that in some cases such ethic...",123
2,6853f700fcdfd17d7f675d57,Pretrained Transformers for Simple Question Answering over Knowledge Graphs,pretrained transformers for simple question answering over knowledge graphs,Answering simple questions over knowledge graphs is a well-studied problem in\nquestion answering. Previous approaches for this task built on recurrent and\nconvolutional neural network based architectu...,answering simple questions over knowledge graphs is a well studied problem in question answering. previous approaches for this task built on recurrent and convolutional neural network based architectu...,pretrained transformers for simple question answering over knowledge graphs. answering simple questions over knowledge graphs is a well studied problem in question answering. previous approaches for this task built on recurrent and convolutional neural network based architectures that use pretrained...,88


## EMBEDDING WITH MINILM

In [17]:
class MiniLMEmbedder:
    def __init__(self, source_collection, target_collection, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.source = source_collection
        self.target = target_collection

        # Load the tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()

        # Set device to GPU if available, otherwise CPU
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        print(f"Using device: {self.device}")

    def embed_text(self, text):
        """
        Generate CLS-based embedding from the given text using the MiniLM model.
        """
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding="max_length"
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)
            cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
        
        return cls_embedding.tolist()

    def process_and_save_embeddings(self, batch_size=64):
        """
        Process and generate embeddings for documents in batches, skipping already processed ones.
        Save the embeddings to the target MongoDB collection.
        """
        total_docs = self.source.count_documents({"is_processed": True})
        print(f"Starting embedding process for {total_docs} documents.")
        pbar = tqdm(total=total_docs)

        for i in range(0, total_docs, batch_size):
            batch = list(self.source.find({"is_processed": True}).skip(i).limit(batch_size))
            batch_ids = [doc["_id"] for doc in batch]

            # Check which documents already have embeddings
            existing_ids = set(
                doc["_id"] for doc in self.target.find({"_id": {"$in": batch_ids}}, {"_id": 1})
            )
            new_docs = [doc for doc in batch if doc["_id"] not in existing_ids]

            docs_to_insert = []
            for doc in new_docs:
                text = doc.get("combined_text", "").strip()
                if not text:
                    continue

                embedding = self.embed_text(text)
                docs_to_insert.append({
                    "_id": doc["_id"],
                    "doc_id": doc.get("id"),
                    "combined_text": text,
                    "embedding": embedding,
                    "model": "All-MiniLM-L6-v2" 
                })

            if docs_to_insert:
                try:
                    self.target.insert_many(docs_to_insert, ordered=False)
                except Exception as e:
                    print(f"Error inserting batch: {e}")

            pbar.update(len(batch))

        pbar.close()
        print("Embedding process completed.")


In [18]:
def connect_to_mongodb(uri="mongodb://localhost:27017", db_name="arxiv_db"):
    client = MongoClient(uri)
    db = client[db_name]
    return db``

db = connect_to_mongodb()
source_collection = db["170k_papers_processed"]
target_collection = db["170k_embedding_MiniLM"]

# embedding process
embedder = MiniLMEmbedder(source_collection, target_collection)
embedder.process_and_save_embeddings(batch_size=32)  


Using device: cuda
Starting embedding process for 179222 documents.


100%|██████████| 179222/179222 [58:34<00:00, 51.00it/s]

Embedding process completed.


In [ ]:
samples = list(target_collection.find({}).limit(3))

results = []
for doc in samples:
    result = {
        "_id": doc["_id"],
        "doc_id": doc.get("doc_id", ""),
        "combined_text": doc.get("combined_text", "")[:300] + "..." if len(doc.get("combined_text", "")) > 300 else doc.get("combined_text", ""),
        "embedding_preview": str(doc.get("embedding", [])[:5]) + "..." 
    }
    results.append(result)

df_samples = pd.DataFrame(results)

pd.set_option("display.max_colwidth", None)
df_samples

,_id,doc_id,combined_text,embedding_preview
0,6853f700fcdfd17d7f675d55,http://arxiv.org/abs/2001.12004v2,"neural mmo v1.3 a massively multiagent game environment for training and evaluating neural networks. progress in multiagent intelligence research is fundamentally limited by the number and quality of environments available for study. in recent years, simulated games have become a dominant research p...","[0.12290927767753601, -0.11322644352912903, -0.05653366073966026, -0.016796717420220375, 0.00520721822977066]..."
1,6853f700fcdfd17d7f675d56,http://arxiv.org/abs/2001.11990v2,"deontological ethics by monotonicity shape constraints. we demonstrate how easy it is for modern machine learned systems to violate common deontological ethical principles and social norms such as favor the less fortunate, and do not penalize good attributes. we propose that in some cases such ethic...","[-0.141126811504364, 0.04398803412914276, 0.08135070651769638, 0.018377481028437614, -0.26154109835624695]..."
2,6853f700fcdfd17d7f675d57,http://arxiv.org/abs/2001.11985v1,pretrained transformers for simple question answering over knowledge graphs. answering simple questions over knowledge graphs is a well studied problem in question answering. previous approaches for this task built on recurrent and convolutional neural network based architectures that use pretrained...,"[-0.35447102785110474, -0.018046680837869644, 0.19135552644729614, 0.1426708698272705, -0.08808214217424393]..."


## RECOMMENDER USING LATENT SEMATIC INDEXING (LSI)

In [ ]:
# hubungkan ke koleksi yang SUDAH diproses
client   = MongoClient("mongodb://localhost:27017")
col_proc = client["arxiv_db"]["170k_papers_processed"]

# ambil semua combined_text + simpan _id untuk mapping 
cursor = col_proc.find({"is_processed": True},
                       {"_id": 1, "combined_text": 1})
doc_ids, texts = [], []
for d in tqdm(cursor, desc="Fetch texts"):
    doc_ids.append(d["_id"])
    texts.append(d["combined_text"])

# TF‑IDF + LSI (SVD)
vectorizer = TfidfVectorizer(stop_words="english",
                             max_df=0.8, min_df=5)
X_tfidf = vectorizer.fit_transform(texts)

svd = TruncatedSVD(n_components=100, random_state=42)
lsi = make_pipeline(svd, Normalizer(copy=False))
X_lsi = lsi.fit_transform(X_tfidf).astype("float32")  # shape: (N, 100)

# simpan artefact
np.save("doc_vectors.npy", X_lsi)          # memory‑mapped file
joblib.dump(vectorizer, "tfidf.pkl")
joblib.dump(lsi,        "lsi.pkl")
joblib.dump(doc_ids,    "doc_ids.pkl")
print(" LSI artefacts saved.")


Fetch texts: 179222it [00:01, 97557.99it/s] 


✅  LSI artefacts saved.


In [104]:
# load artefacts sekali di awal aplikasi
doc_vecs  = np.load("lsi/doc_vectors.npy", mmap_mode="r")   # read‑only memmap
tfidf     = joblib.load("lsi/tfidf.pkl")
lsi       = joblib.load("lsi/lsi.pkl")
doc_ids   = joblib.load("lsi/doc_ids.pkl")

mongo     = MongoClient("mongodb://localhost:27017")
col_proc  = mongo["arxiv_db"]["170k_papers_processed"]

def recommend_lsi(query: str, top_k: int = 5):
    # encode query → LSI space
    q_vec  = tfidf.transform([query])
    q_lsi  = lsi.transform(q_vec)

    # cosine = dot product karena semua vec sudah dinormalisasi
    scores = np.dot(doc_vecs, q_lsi.T).ravel()

    # ambil top‑k
    idx    = scores.argsort()[-top_k:][::-1]
    results = []
    for i in idx:
        doc = col_proc.find_one({"_id": doc_ids[i]},
                                {"title":1, "authors":1,
                                 "published":1, "primary_category":1, "_id": 1,
                                 "pdf_url":1, "summary":1})
        results.append((doc, float(scores[i])))
    return results


In [105]:
for n,(doc,score) in enumerate(recommend_lsi(
        "Object Detection with YOLO"), 1):
    print(f"\n== Article #{n} ==")
    print(f"Title : {doc['title']}")
    print(f"Score : {score:.4f}")
    print(f"Authors   : {doc['authors'][:3]} ...")
    print(f"Published : {doc['published']}")
    print(f"Category  : {doc['primary_category']}")
    print(f"PDF URL   : {doc['pdf_url']}")
    print(f"Summary   : {doc['summary'][:300]}...")



== Article #1 ==
Title : SWA Object Detection
Score : 0.9590
Authors   : Hao ...
Published : 2020-12-23
Category  : cs.CV
PDF URL   : http://arxiv.org/pdf/2012.12645v3
Summary   : Do you want to improve 1.0 AP for your object detector without any inference
cost and any change to your detector? Let us tell you such a recipe. It is
surprisingly simple: train your detector for an extra 12 epochs using cyclical
learning rates and then average these 12 checkpoints as your final de...

== Article #2 ==
Title : Visibility Guided NMS: Efficient Boosting of Amodal Object Detection in Crowded Traffic Scenes
Score : 0.9321
Authors   : Nil ...
Published : 2020-06-15
Category  : cs.CV
PDF URL   : http://arxiv.org/pdf/2006.08547v1
Summary   : Object detection is an important task in environment perception for
autonomous driving. Modern 2D object detection frameworks such as Yolo, SSD or
Faster R-CNN predict multiple bounding boxes per object that are refined using
Non-Maximum-Suppression (NMS) to s

In [51]:
import gradio as gr

def gradio_recommend(query, min_year, max_year, sort_by, dark_mode):
    results = recommend_lsi(query, 20)
    if min_year > max_year:
        min_year, max_year = max_year, min_year

    filtered = []
    for doc, score in results:
        year_txt = doc.get("published", "")[:4]
        if not year_txt.isdigit():
            continue
        year = int(year_txt)
        if not (min_year <= year <= max_year):
            continue
        filtered.append((doc, score, year))

    # Sorting logic
    if sort_by == "Score":
        filtered.sort(key=lambda x: x[1], reverse=True)
    elif sort_by == "Year (Newest First)":
        filtered.sort(key=lambda x: x[2], reverse=True)
    elif sort_by == "Year (Oldest First)":
        filtered.sort(key=lambda x: x[2])

    cards_html, counter = [], 0
    for doc, score, year in filtered:
        counter += 1
        title = doc.get("title", "Untitled")
        authors = ', '.join(doc.get("authors", [])[:4])
        if len(doc.get("authors", [])) > 4:
            authors += " et al."
        pdf = doc.get("pdf_url", "#")
        summary = doc.get("summary", "").replace("\n", " ")[:300] + "…"

        cards_html.append(f"""
        <article class="card">
          <h3><a href="{pdf}" target="_blank">{title}</a></h3>
          <p class="meta">{authors} · {year} · Score {score:.3f}</p>
          <p class="abs">{summary}</p>
          <a class="btn" href="{pdf}" target="_blank">Read Paper</a>
        </article>
        """)

    if not cards_html:
        return """
        <div class="empty">
          <h3>No results found</h3>
          <p>Try different keywords or year range.</p>
        </div>
        """

    header = f"""
    <div class="summary">{counter} papers for “{query}” ({min_year}–{max_year})</div>
    """
    # Add dark mode wrapper if active
    final_html = header + "".join(cards_html)
    if dark_mode:
        final_html = f"<div class='dark'>{final_html}</div>"
    return final_html

# CSS 
custom_css = """
body { font-family: -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif; }
.gradio-container { max-width: 980px; margin: 0 auto; }
h1.title { font-size: 2.4rem; font-weight: 600; text-align: center; margin: 10px 0 32px; }
.summary { background: #f1f3f4; border: 1px solid #e0e0e0; border-radius: 6px; padding: 12px 16px; margin-bottom: 22px; }
.card { background: #fff; border: 1px solid #e0e0e0; border-radius: 8px; padding: 18px 20px; margin-bottom: 18px;
        box-shadow: 0 1px 3px rgb(0 0 0 / 8%); }
.card h3 { margin: 0 0 8px; font-size: 1.1rem; line-height: 1.35; }
.card a { color: #1a73e8; text-decoration: none; font-weight: 500; }
.card a:hover { text-decoration: underline; }
.meta { color: #5f6368; font-size: .87rem; margin: 0 0 12px; }
.abs { color: #3c4043; font-size: .95rem; line-height: 1.45; margin: 0 0 14px; }
.btn { display: inline-block; padding: 6px 14px; border: 1px solid #1a73e8; border-radius: 4px; font-size: .90rem;
       text-decoration: none; color: #1a73e8; transition: background .15s; }
.btn:hover { background: #e8f0fe; }
.empty { text-align: center; padding: 48px 20px; color: #5f6368; }

/* Dark mode */
.dark .card { background: #1e1e1e; border-color: #333; }
.dark .summary { background: #2c2c2c; border-color: #444; color: #eee; }
.dark h3 a, .dark .btn { color: #8ab4f8; border-color: #8ab4f8; }
.dark .btn:hover { background: #1a1a1a; }
.dark .meta, .dark .abs { color: #ccc; }
"""

# UI 
with gr.Blocks(css=custom_css) as demo:
    gr.Markdown('<h1 class="title">ArXiv Paper Recommender</h1>')

    query_input = gr.Textbox(
        placeholder="Enter research topic or keywords",
        label="Search Paper",
        lines=1
    )

    with gr.Row():
        min_year_slider = gr.Slider(2021, 2025, 2021, step=1, label="From Year")
        max_year_slider = gr.Slider(2021, 2025, 2025, step=1, label="To Year")

    sort_dropdown = gr.Dropdown(
        choices=["Score", "Year (Newest First)", "Year (Oldest First)"],
        value="Score",
        label="Sort By"
    )

    search_btn = gr.Button("Search", variant="primary")

    output_html = gr.HTML("<div class='empty'>Enter a query to begin.</div>")

    # Event binding
    search_btn.click(
        fn=gradio_recommend,
        inputs=[query_input, min_year_slider, max_year_slider, sort_dropdown, dark_mode_toggle],
        outputs=output_html
    )
    query_input.submit(
        fn=gradio_recommend,
        inputs=[query_input, min_year_slider, max_year_slider, sort_dropdown, dark_mode_toggle],
        outputs=output_html
    )

if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


## MINI-LM WITH COSINE

In [67]:
emb_col         = db["170k_embedding_MiniLM"]      # vektor
meta_col        = db["170k_papers_processed"]      # metadata

#Load MiniLM model untuk meng‑embed query
MODEL_NAME      = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer       = AutoTokenizer.from_pretrained(MODEL_NAME)
model           = AutoModel.from_pretrained(MODEL_NAME).eval().to(
                     torch.device("cuda" if torch.cuda.is_available() else "cpu"))

def embed_query(text: str) -> np.ndarray:
    """Encode text → CLS embedding (unit‑length)."""
    with torch.no_grad():
        inputs  = tokenizer(text, return_tensors="pt",
                            padding="max_length", truncation=True,
                            max_length=512).to(model.device)
        output  = model(**inputs).last_hidden_state[:, 0, :]          # CLS
        vec     = output.squeeze().cpu().numpy()
    # normalisasi agar ||vec||=1 (penting untuk cosine = dot)
    return vec / np.linalg.norm(vec)

#    (≈170 000 × 768 floats ≈ 500 MB float32 
doc_ids, doc_vecs = [], []
for doc in tqdm.tqdm(emb_col.find({}, {"_id":1, "embedding":1}),
                     desc="Load MiniLM embeddings"):
    doc_ids.append(doc["_id"])
    v = np.asarray(doc["embedding"], dtype="float32")
    v = v / np.linalg.norm(v)       
    doc_vecs.append(v)

doc_vecs = np.vstack(doc_vecs)  # shape (N, 768)

# SIMPAN KE FILE — INI YANG PENTING!
np.save("minilm/doc_vectors.npy", doc_vecs)
joblib.dump(doc_ids, "minilm/doc_ids.pkl")

Load MiniLM embeddings: 179222it [00:33, 5400.12it/s] 


['minilm/doc_ids.pkl']

In [120]:
def recommend_minilm(query: str, top_k: int = 5):
    """Return list[(metadata, score)] untuk query tertentu."""
    q_vec = embed_query(query).astype("float32")  # (384,) untuk MiniLM

    # Load vektor MiniLM secara lokal agar tidak bentrok dengan LSI
    minilm_doc_vecs = np.load("minilm/doc_vectors.npy", mmap_mode="r")  # (N, 384)

    # Safety check dimensi
    if q_vec.shape[0] != minilm_doc_vecs.shape[1]:
        raise ValueError(f"Dimensi q_vec ({q_vec.shape[0]}) ≠ minilm_doc_vecs ({minilm_doc_vecs.shape[1]})")

    # cosine similarity (karena semua sudah di-normalisasi)
    scores = minilm_doc_vecs @ q_vec  # shape (N,)

    # ambil indeks skor tertinggi
    top_idx = np.argpartition(scores, -top_k)[-top_k:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]  # descending

    results = []
    for i in top_idx:
        meta = meta_col.find_one(
            {"_id": doc_ids[i]},
            {
                "title": 1,
                "authors": 1,
                "published": 1,
                "_id": 1,
                "primary_category": 1,
                "pdf_url": 1,
                "summary": 1
            }
        )
        if meta:
            results.append((meta, float(scores[i])))
        else:
            print("WARNING: Metadata not found for _id:", doc_ids[i])

    return results


In [88]:
print("Total embeddings:", len(doc_ids))
print("Contoh ID:", doc_ids[0])
print("Coba cari di metadata:")
print(meta_col.find_one({"_id": doc_ids[0]}))

Total embeddings: 179222
Contoh ID: 6853f700fcdfd17d7f675d55
Coba cari di metadata:
{'_id': ObjectId('6853f700fcdfd17d7f675d55'), 'id': 'http://arxiv.org/abs/2001.12004v2', 'title': 'Neural MMO v1.3: A Massively Multiagent Game Environment for Training and Evaluating Neural Networks', 'summary': 'Progress in multiagent intelligence research is fundamentally limited by the\nnumber and quality of environments available for study. In recent years,\nsimulated games have become a dominant research platform within reinforcement\nlearning, in part due to their accessibility and interpretability. Previous\nworks have targeted and demonstrated success on arcade, first person shooter\n(FPS), real-time strategy (RTS), and massive online battle arena (MOBA) games.\nOur work considers massively multiplayer online role-playing games (MMORPGs or\nMMOs), which capture several complexities of real-world learning that are not\nwell modeled by any other game genre. We present Neural MMO, a massively\nmul

In [121]:
results = recommend_minilm("object detection with YOLO", top_k=5)
for n, (doc, score) in enumerate(results, 1):
    print(f"\n== #{n} | Score: {score:.3f}")
    print("Title :", doc["title"])
    print("Year  :", doc["published"][:4], "| Category:", doc["primary_category"])
    print("PDF   :", doc["pdf_url"])
    print("Summary:", doc["summary"][:300] + "…")


== #1 | Score: 0.897
Title : YOLO Evolution: A Comprehensive Benchmark and Architectural Review of YOLOv12, YOLO11, and Their Previous Versions
Year  : 2024 | Category: cs.CV
PDF   : http://arxiv.org/pdf/2411.00201v4
Summary: This study presents a comprehensive benchmark analysis of various YOLO (You
Only Look Once) algorithms. It represents the first comprehensive experimental
evaluation of YOLOv3 to the latest version, YOLOv12, on various object
detection challenges. The challenges considered include varying object siz…

== #2 | Score: 0.865
Title : Real Time Object Detection System with YOLO and CNN Models: A Review
Year  : 2022 | Category: cs.CV
PDF   : http://arxiv.org/pdf/2208.00773v1
Summary: The field of artificial intelligence is built on object detection techniques.
YOU ONLY LOOK ONCE (YOLO) algorithm and it's more evolved versions are briefly
described in this research survey. This survey is all about YOLO and
convolution neural networks (CNN)in the direction of real time o

In [ ]:
# import numpy as np
# import pandas as pd
# from collections import defaultdict
# from typing import Callable, Dict, Iterable, List, Sequence, Tuple

# # ------------------------------------------------------------
# #  Helpers ‑‑ metrik retrieval klasik
# # ------------------------------------------------------------
# def _precision_at_k(retrieved: Sequence[str], relevant: set, k: int) -> float:
#     retrieved_k = retrieved[:k]
#     if not retrieved_k:                       # tidak ada hasil
#         return 0.0
#     hit = sum(doc_id in relevant for doc_id in retrieved_k)
#     return hit / k

# def _recall_at_k(retrieved: Sequence[str], relevant: set, k: int) -> float:
#     if not relevant:                          # query tanpa dok relavan → recall tidak terdefinisi
#         return 0.0
#     retrieved_k = retrieved[:k]
#     hit = sum(doc_id in relevant for doc_id in retrieved_k)
#     return hit / len(relevant)

# def _dcg_at_k(retrieved: Sequence[str], relevant: set, k: int) -> float:
#     dcg = 0.0
#     for i, doc_id in enumerate(retrieved[:k], 1):          # posisi dimulai dari 1
#         rel = 1.0 if doc_id in relevant else 0.0
#         if rel:
#             dcg += rel / np.log2(i + 1)
#     return dcg

# def _ndcg_at_k(retrieved: Sequence[str], relevant: set, k: int) -> float:
#     ideal_hits = min(len(relevant), k)
#     if ideal_hits == 0:
#         return 0.0
#     idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
#     return _dcg_at_k(retrieved, relevant, k) / idcg

# # ------------------------------------------------------------
# #  Fungsi utama
# # ------------------------------------------------------------
# def compare_recommenders(
#     query_list: Iterable[str],
#     qrel_dict: Dict[str, set],
#     ks: Sequence[int] = (5, 10),
#     recommenders: Dict[str, Callable[[str, int], List[Tuple[dict, float]]]] = None,
# ) -> pd.DataFrame:
#     """
#     Bandingkan beberapa fungsi rekomendasi dalam sekali jalan.

#     Parameters
#     ----------
#     query_list : iterable[str]
#         Daftar query yang akan dievaluasi.
#     qrel_dict : dict[str, set[str]]
#         Ground‑truth: mapping query → set dokumen relevan (ObjectId atau string).
#     ks : iterable[int], default (5, 10)
#         Nilai‑K yang ingin dihitung (Precision@K, Recall@K, NDCG@K).
#     recommenders : dict[str, callable], default None
#         Mapping `nama_model` → fungsi_rekomendasi(query, top_k)
#         Jika None maka otomatis menggunakan:
#            {"LSI": recommend_lsi, "MiniLM": recommend_minilm}

#     Returns
#     -------
#     pandas.DataFrame  (rows = model×K, cols = metrik)
#     """
#     if recommenders is None:
#         recommenders = {
#             "LSI": recommend_lsi,
#             "MiniLM": recommend_minilm,
#         }

#     # struktur penampung hasil
#     rows = defaultdict(list)

#     for model_name, rec_fn in recommenders.items():
#         for k in ks:
#             precs, recalls, ndcgs = [], [], []

#             for q in query_list:
#                 retrieved = [meta["_id"] for meta, _ in rec_fn(q, top_k=k)]
#                 relevant  = qrel_dict.get(q, set())

#                 precs.append(_precision_at_k(retrieved, relevant, k))
#                 recalls.append(_recall_at_k(retrieved, relevant, k))
#                 ndcgs.append(_ndcg_at_k(retrieved, relevant, k))

#             # rata‑rata antar query
#             rows["model"].append(model_name)
#             rows["k"].append(k)
#             rows["precision"].append(np.mean(precs))
#             rows["recall"].append(np.mean(recalls))
#             rows["ndcg"].append(np.mean(ndcgs))

#     return pd.DataFrame(rows)

In [126]:
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import Callable, Dict, Iterable, List, Sequence, Tuple

# ------------------------------------------------------------
#  Helpers – metrik retrieval klasik & tambahan
# ------------------------------------------------------------
def _precision_at_k(retrieved: Sequence[str], relevant: set, k: int) -> float:
    retrieved_k = retrieved[:k]
    if not retrieved_k:                       # tidak ada hasil
        return 0.0
    hit = sum(doc_id in relevant for doc_id in retrieved_k)
    return hit / k


def _recall_at_k(retrieved: Sequence[str], relevant: set, k: int) -> float:
    if not relevant:                          # query tanpa dok relevan → recall tidak terdefinisi
        return 0.0
    retrieved_k = retrieved[:k]
    hit = sum(doc_id in relevant for doc_id in retrieved_k)
    return hit / len(relevant)


def _dcg_at_k(retrieved: Sequence[str], relevant: set, k: int) -> float:
    dcg = 0.0
    for i, doc_id in enumerate(retrieved[:k], 1):          # posisi dimulai dari 1
        rel = 1.0 if doc_id in relevant else 0.0
        if rel:
            dcg += rel / np.log2(i + 1)
    return dcg


def _ndcg_at_k(retrieved: Sequence[str], relevant: set, k: int) -> float:
    ideal_hits = min(len(relevant), k)
    if ideal_hits == 0:
        return 0.0
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return _dcg_at_k(retrieved, relevant, k) / idcg


# ---------- metrik tambahan ---------------------------------
def _error_rate_at_k(precision: float) -> float:
    """Karena kelas positif = 'relevan',
       Error Rate = 1 – Precision (pada top‑k)."""
    return 1.0 - precision


def _mse_at_k(retrieved: Sequence[str], relevant: set, k: int) -> float:
    """
    Mean‑Squared Error antara prediksi biner (1 = di‑retrieve)
    dan label relevansi biner (1 = relevan). Untuk posisi <= k.
    Karena prediksi bernilai 1 utk setiap dok pada top‑k,
    MSE = Error Rate (tetap kami pisahkan agar eksplisit).
    """
    precision = _precision_at_k(retrieved, relevant, k)
    return _error_rate_at_k(precision)


def _mae_at_k(retrieved: Sequence[str], relevant: set, k: int) -> float:
    """
    Mean‑Absolute Error – dengan skema yang sama seperti MSE,
    ternyata identik dengan Error Rate. Disediakan demi
    kelengkapan laporan.
    """
    return _mse_at_k(retrieved, relevant, k)


def _accuracy_at_k(precision: float) -> float:
    """Dengan hanya mempertimbangkan top‑k dokumen,
    accuracy ≡ precision (karena setiap instance = dokumen)."""
    return precision


def _f1_at_k(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# ------------------------------------------------------------
#  Fungsi utama
# ------------------------------------------------------------
def compare_recommenders(
    query_list: Iterable[str],
    qrel_dict: Dict[str, set],
    ks: Sequence[int] = (5, 10),
    recommenders: Dict[str, Callable[[str, int], List[Tuple[dict, float]]]] | None = None,
) -> pd.DataFrame:
   
    if recommenders is None:
        recommenders = {
            "LSI": recommend_lsi,
            "MiniLM": recommend_minilm,
        }
   
    rows = defaultdict(list)

    for model_name, rec_fn in recommenders.items():
        for k in ks:
            precs, recalls, ndcgs = [], [], []
            errs, mses, maes, accs, f1s = [], [], [], [], []

            for q in query_list:
                retrieved_pairs = rec_fn(q, top_k=k)
                retrieved_ids   = [meta["_id"] for meta, _ in retrieved_pairs]
                relevant        = qrel_dict.get(q, set())

                p  = _precision_at_k(retrieved_ids, relevant, k)
                r  = _recall_at_k(retrieved_ids, relevant, k)
                n  = _ndcg_at_k(retrieved_ids, relevant, k)
                er = _error_rate_at_k(p)
                ms = _mse_at_k(retrieved_ids, relevant, k)
                ma = _mae_at_k(retrieved_ids, relevant, k)
                ac = _accuracy_at_k(p)
                f1 = _f1_at_k(p, r)

                precs.append(p); recalls.append(r); ndcgs.append(n)
                errs.append(er); mses.append(ms); maes.append(ma)
                accs.append(ac); f1s.append(f1)

            # rata‑rata antar query
            rows["model"].append(model_name)
            rows["k"].append(k)
            rows["precision"].append(np.mean(precs))
            rows["recall"].append(np.mean(recalls))
            rows["ndcg"].append(np.mean(ndcgs))
            rows["error_rate"].append(np.mean(errs))
            rows["mse"].append(np.mean(mses))
            rows["mae"].append(np.mean(maes))
            rows["accuracy"].append(np.mean(accs))
            rows["f1"].append(np.mean(f1s))

    return pd.DataFrame(rows)

In [127]:
import pandas as pd

query_list = [
    "graph convolutional networks for drug discovery",
    "domain adaptation in medical imaging",
    "reinforcement learning for robot manipulation",
    "few‑shot text classification",
    "adversarial attacks on vision transformers",
]

qrel_dict = {}
for q in query_list:
    lsi_results   = recommend_lsi(q, top_k=15)
    minilm_results = recommend_minilm(q, top_k=15)

    # kumpulkan _id dokumen dari kedua hasil
    relevant_ids = {meta["_id"] for meta, _ in lsi_results}
    relevant_ids.update(meta["_id"] for meta, _ in minilm_results)

    qrel_dict[q] = relevant_ids

metrics_df = compare_recommenders(
    query_list,          # daftar query
    qrel_dict,           # ground‑truth relevansi
    ks=(5, 10, 20),      # nilai‑K yang dievaluasi
)

print("\n==== Hasil evaluasi ====")
print(metrics_df.to_string(index=False))



==== Hasil evaluasi ====
 model  k  precision   recall     ndcg  error_rate  mse  mae  accuracy       f1
   LSI  5       1.00 0.178013 1.000000        0.00 0.00 0.00      1.00 0.302063
   LSI 10       1.00 0.356026 1.000000        0.00 0.00 0.00      1.00 0.524677
   LSI 20       0.75 0.534039 0.832546        0.25 0.25 0.25      0.75 0.623231
MiniLM  5       1.00 0.178013 1.000000        0.00 0.00 0.00      1.00 0.302063
MiniLM 10       1.00 0.356026 1.000000        0.00 0.00 0.00      1.00 0.524677
MiniLM 20       0.76 0.542039 0.839014        0.24 0.24 0.24      0.76 0.632120


## GRADIO INTERFACE

In [13]:
meta_col    = db["170k_papers_processed"]

# MiniLM model (hanya untuk query)
MODEL_NAME  = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer   = AutoTokenizer.from_pretrained(MODEL_NAME)
device = torch.device("cpu")  # paksa CPU
model = AutoModel.from_pretrained(MODEL_NAME).eval().to(device)

def embed_query(text: str) -> np.ndarray:
    """MiniLM CLS‑embedding, sudah dinormalisasi."""
    with torch.no_grad():
        inp   = tokenizer(text, return_tensors="pt",
                          padding="max_length", truncation=True,
                          max_length=512).to(model.device)
        vec   = model(**inp).last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    return vec / np.linalg.norm(vec)

#LSI helper
def recommend_lsi(query: str, top_k: int = 20):
    vecs   = np.load("lsi/doc_vectors.npy", mmap_mode="r")
    tfidf  = joblib.load("lsi/tfidf.pkl")
    lsi    = joblib.load("lsi/lsi.pkl")
    ids    = joblib.load("lsi/doc_ids.pkl")

    q_lsi  = lsi.transform(tfidf.transform([query]))         # (1,100)
    scores = vecs @ q_lsi.T                                  # (N,1)
    top    = scores.ravel().argsort()[-top_k:][::-1]

    return [(ids[i], float(scores[i])) for i in top]

# MiniLM helper
def recommend_minilm(query: str, top_k: int = 20):
    vecs  = np.load("minilm/doc_vectors.npy", mmap_mode="r")
    ids   = joblib.load("minilm/doc_ids.pkl")

    q_vec = embed_query(query).astype("float32")             # (384,)
    scores= vecs @ q_vec                                     # (N,)
    top   = scores.argsort()[-top_k:][::-1]

    return [(ids[i], float(scores[i])) for i in top]

#  Gradio callback 
def gradio_recommend(query, min_year, max_year, sort_by, rec_type):
    pairs = recommend_lsi(query)  if rec_type == "LSI" else recommend_minilm(query)

    # filter tahun
    filtered = []
    for _id, score in pairs:
        doc = meta_col.find_one({"_id": _id},
                 {"_id":1, "title":1,"authors":1,"published":1,
                  "primary_category":1,"pdf_url":1,"summary":1})
        if not doc: continue
        y = int(doc.get("published","")[:4] or 0)
        if min_year <= y <= max_year:
            filtered.append((doc, score, y))

    # sort
    if sort_by == "Score":
        filtered.sort(key=lambda x: x[1], reverse=True)
    elif sort_by == "Year (Newest First)":
        filtered.sort(key=lambda x: x[2], reverse=True)
    else:
        filtered.sort(key=lambda x: x[2])

    # html
    if not filtered:
        return ("<div class='empty'><h3>No results found</h3>"
                "<p>Try different keywords or year range.</p></div>")

    items = []
    for doc, score, y in filtered:
        title   = doc.get("title","Untitled")
        authors = ', '.join(doc.get("authors",[])[:4]) + (" et al." if len(doc.get("authors",[]))>4 else "")
        pdf     = doc.get("pdf_url","#")
        summ    = doc.get("summary","").replace("\n"," ")[:300] + "…"
        cat     = doc.get("primary_category","")
        items.append(f"""
        <article class="card">
           <h3><a href="{pdf}" target="_blank">{title}</a></h3>
           <p class="meta">{authors} · {y} · {cat} · Score {score:.3f}</p>
           <p class="abs">{summ}</p>
           <a class="btn" href="{pdf}" target="_blank">Read Paper</a>
        </article>""")

    header = (f"<div class='summary'>{len(filtered)} papers for "
              f"“{query}” ({min_year}–{max_year})</div>")
    return header + "".join(items)

# CSS
custom_css = """
body { font-family: -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif; }
.gradio-container { max-width: 980px; margin: 0 auto; }
h1.title { font-size: 2.4rem; font-weight: 600; text-align: center; margin: 10px 0 32px; }
.summary { background: #f1f3f4; border: 1px solid #e0e0e0; border-radius: 6px; padding: 12px 16px; margin-bottom: 22px; }
.card { background: #fff; border: 1px solid #e0e0e0; border-radius: 8px; padding: 18px 20px; margin-bottom: 18px;
        box-shadow: 0 1px 3px rgb(0 0 0 / 8%); }
.card h3 { margin: 0 0 8px; font-size: 1.1rem; line-height: 1.35; }
.card a { color: #1a73e8; text-decoration: none; font-weight: 500; }
.card a:hover { text-decoration: underline; }
.meta { color: #5f6368; font-size: .87rem; margin: 0 0 12px; }
.abs { color: #3c4043; font-size: .95rem; line-height: 1.45; margin: 0 0 14px; }
.btn { display: inline-block; padding: 6px 14px; border: 1px solid #1a73e8; border-radius: 4px; font-size: .90rem;
       text-decoration: none; color: #1a73e8; transition: background .15s; }
.btn:hover { background: #e8f0fe; }
.empty { text-align: center; padding: 48px 20px; color: #5f6368; }
"""

# UI
with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("<h1 class='title'>ArXiv Paper Recommender</h1>")

    query = gr.Textbox(label="Search Paper", placeholder="Enter research topic or keywords")
    with gr.Row():
        yr_min = gr.Slider(2021, 2025, value=2021, step=1, label="From Year")
        yr_max = gr.Slider(2021, 2025, value=2025, step=1, label="To Year")
    sort   = gr.Dropdown(["Score","Year (Newest First)","Year (Oldest First)"],
                         value="Score", label="Sort By")
    recsel = gr.Dropdown(["LSI","MiniLM"], value="LSI", label="Recommender Type")
    btn    = gr.Button("Search", variant="primary")
    out    = gr.HTML("<div class='empty'>Enter a query to begin.</div>")

    btn.click(gradio_recommend,
              inputs=[query, yr_min, yr_max, sort, recsel],
              outputs=out)
    query.submit(gradio_recommend,
              inputs=[query, yr_min, yr_max, sort, recsel],
              outputs=out)

if __name__ == "__main__":
    demo.launch(show_error=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
